
# Hurwitz-Kriterium — Hauptabschnittsdeterminanten

Dieses Notebook bestimmt für ein charakteristisches Polynom

$$
p(s) = a_0 s^n + a_1 s^{n-1} + \dots + a_{n-1} s + a_n
$$

die **Hurwitz-Matrix** und daraus die **Hauptabschnittsdeterminanten**
$\Delta_1, \Delta_2, \dots, \Delta_n$.

**Stabilitätsaussage (Hurwitz-Kriterium):**
Das System ist genau dann stabil (alle Wurzeln von $p(s)$ liegen in der linken
Halbebene), wenn

- $a_0 > 0$ und
- $\Delta_k > 0$ für alle $k = 1, \dots, n$.

Die Koeffizienten können sowohl **numerisch** (float/int) als auch
**symbolisch** (z. B. mit einem Parameter $K$) übergeben werden — dank `sympy`.


In [1]:
import sympy as sp

sp.init_printing()


## 1. Aufbau der Hurwitz-Matrix

Koeffizienten werden als Liste `[a0, a1, ..., an]` übergeben
(also **absteigend** nach Potenzen von $s$, beginnend bei $a_0$, dem
Koeffizienten von $s^n$).

Die Hurwitz-Matrix $H \in \mathbb{R}^{n \times n}$ ist definiert über

$$
H_{ij} = \begin{cases} a_{2j-i} & \text{falls } 0 \le 2j-i \le n \\ 0 & \text{sonst} \end{cases}
\qquad i,j = 1, \dots, n


In [2]:
def hurwitz_matrix(coeffs):
    '''
    Baut die Hurwitz-Matrix zu einem charakteristischen Polynom auf.

    Parameter
    ----------
    coeffs : list
        Koeffizienten [a0, a1, ..., an] des Polynoms
        p(s) = a0*s^n + a1*s^(n-1) + ... + an  (a0 = führender Koeffizient)

    Rückgabe
    --------
    sympy.Matrix : n x n Hurwitz-Matrix
    '''
    a = list(coeffs)
    n = len(a) - 1  # Grad des Polynoms

    def coeff(k):
        # a_k, mit a_k = 0 fuer k ausserhalb [0, n]
        if 0 <= k <= n:
            return sp.sympify(a[k])
        return sp.Integer(0)

    H = sp.zeros(n, n)
    for i in range(1, n + 1):
        for j in range(1, n + 1):
            H[i - 1, j - 1] = coeff(2 * j - i)

    return H


## 2. Hauptabschnittsdeterminanten

Die Hauptabschnittsdeterminante $\Delta_k$ ist die Determinante der
oberen linken $k \times k$-Teilmatrix der Hurwitz-Matrix.


In [3]:
def hauptabschnittsdeterminanten(coeffs, vereinfachen=True):
    '''
    Berechnet alle Hauptabschnittsdeterminanten Delta_1 ... Delta_n
    der Hurwitz-Matrix zu einem charakteristischen Polynom.

    Parameter
    ----------
    coeffs : list
        Koeffizienten [a0, a1, ..., an] des Polynoms.
    vereinfachen : bool
        Wenn True, werden die Determinanten (v. a. bei symbolischen
        Koeffizienten) mit sympy.simplify vereinfacht.

    Rückgabe
    --------
    list[sympy.Expr] : [Delta_1, Delta_2, ..., Delta_n]
    '''
    H = hurwitz_matrix(coeffs)
    n = H.shape[0]

    deltas = []
    for k in range(1, n + 1):
        Hk = H[:k, :k]
        det_k = Hk.det()
        if vereinfachen:
            det_k = sp.simplify(det_k)
        deltas.append(det_k)

    return deltas


## 3. Komplette Stabilitätsprüfung

Fasst Hurwitz-Matrix, Hauptabschnittsdeterminanten und die Stabilitätsaussage
übersichtlich zusammen (funktioniert numerisch **und** symbolisch).


In [4]:
def hurwitz_kriterium(coeffs, vereinfachen=True, anzeigen=True):
    '''
    Führt das vollständige Hurwitz-Kriterium für ein charakteristisches
    Polynom aus und gibt die Hauptabschnittsdeterminanten sowie
    (bei rein numerischen Koeffizienten) die Stabilitätsaussage zurück.
    '''
    H = hurwitz_matrix(coeffs)
    deltas = hauptabschnittsdeterminanten(coeffs, vereinfachen=vereinfachen)

    if anzeigen:
        print("Charakteristisches Polynom (Koeffizienten a0..an):", coeffs)
        print("\nHurwitz-Matrix:")
        sp.pprint(H)
        print()
        for k, d in enumerate(deltas, start=1):
            print(f"Delta_{k} =")
            sp.pprint(d)
            print()

    # Stabilitätsaussage nur sinnvoll auswertbar, wenn alles numerisch ist
    numerisch = all(sp.sympify(a).is_number for a in coeffs) and \
                all(d.is_number for d in deltas)

    if numerisch:
        a0_positiv = sp.sympify(coeffs[0]) > 0
        alle_delta_positiv = all(d > 0 for d in deltas)
        stabil = bool(a0_positiv and alle_delta_positiv)
        if anzeigen:
            status = "STABIL" if stabil else "NICHT stabil"
            print(f"a0 > 0: {a0_positiv}  |  alle Delta_k > 0: {alle_delta_positiv}")
            print(f"=> System ist {status}.")
        return H, deltas, stabil

    if anzeigen:
        print("Koeffizienten enthalten freie Parameter — Stabilitätsbereich "
              "ergibt sich aus den Bedingungen Delta_k > 0 (siehe oben).")
    return H, deltas, None


## 4. Beispiel 1 — numerisches Polynom

$$
p(s) = s^4 + 4s^3 + 6s^2 + 4s + 1
$$


In [5]:
coeffs_beispiel1 = [1, 4, 6, 4, 1]  # a0, a1, a2, a3, a4

H1, deltas1, stabil1 = hurwitz_kriterium(coeffs_beispiel1)

Charakteristisches Polynom (Koeffizienten a0..an): [1, 4, 6, 4, 1]

Hurwitz-Matrix:
⎡4  4  0  0⎤
⎢          ⎥
⎢1  6  1  0⎥
⎢          ⎥
⎢0  4  4  0⎥
⎢          ⎥
⎣0  1  6  1⎦

Delta_1 =
4

Delta_2 =
20

Delta_3 =
64

Delta_4 =
64

a0 > 0: True  |  alle Delta_k > 0: True
=> System ist STABIL.



## 5. Beispiel 2 — symbolisches Polynom mit Parameter $K$

Für einen Regelkreis mit z. B.

$$
p(s) = s^3 + 3s^2 + 3s + (1 + K)
$$

lässt sich damit direkt der **stabile Bereich für $K$** bestimmen, indem
man die Bedingungen $\Delta_k > 0$ nach $K$ auflöst.


In [9]:
K = sp.symbols('K', real=True)

coeffs_beispiel2 = [1+K, 3, 3, 1]

H2, deltas2, _ = hurwitz_kriterium(coeffs_beispiel2)

print("\nBedingungen fuer Stabilitaet (Delta_k > 0):")
for k, d in enumerate(deltas2, start=1):
    bedingung = sp.solve(sp.Gt(d, 0), K)
    print(f"Delta_{k} > 0  =>  {bedingung}")

Charakteristisches Polynom (Koeffizienten a0..an): [K + 1, 3, 3, 1]

Hurwitz-Matrix:
⎡  3    1  0⎤
⎢           ⎥
⎢K + 1  3  0⎥
⎢           ⎥
⎣  0    3  1⎦

Delta_1 =
3

Delta_2 =
8 - K

Delta_3 =
8 - K

Koeffizienten enthalten freie Parameter — Stabilitätsbereich ergibt sich aus den Bedingungen Delta_k > 0 (siehe oben).

Bedingungen fuer Stabilitaet (Delta_k > 0):
Delta_1 > 0  =>  []
Delta_2 > 0  =>  K < 8
Delta_3 > 0  =>  K < 8


In [10]:
K=sp.symbols('K', real=True)
coeffs_1=[K,5,4,1]
H3, deltas3, _ = hurwitz_kriterium(coeffs_1)

print("\nBedingungen fuer Stabilitaet (Delta_k > 0):")
for k, d in enumerate(deltas3, start=1):
    bedingung = sp.solve(sp.Gt(d, 0), K)
    print(f"Delta_{k} > 0  =>  {bedingung}")

Charakteristisches Polynom (Koeffizienten a0..an): [K, 5, 4, 1]

Hurwitz-Matrix:
⎡5  1  0⎤
⎢       ⎥
⎢K  4  0⎥
⎢       ⎥
⎣0  5  1⎦

Delta_1 =
5

Delta_2 =
20 - K

Delta_3 =
20 - K

Koeffizienten enthalten freie Parameter — Stabilitätsbereich ergibt sich aus den Bedingungen Delta_k > 0 (siehe oben).

Bedingungen fuer Stabilitaet (Delta_k > 0):
Delta_1 > 0  =>  []
Delta_2 > 0  =>  K < 20
Delta_3 > 0  =>  K < 20


## 6. Paket-Integration (Hurwitz + Routh)

Hier nutzen wir die Paketfunktionen statt einer separaten Eigenimplementierung.

- Eingang: Nennerkoeffizienten eines charakteristischen Polynoms
- Ausgabe: Stabilitätsaussage nach Hurwitz und Routh

In [ ]:
from regelungstechnik import hurwitz_kriterium, routh_kriterium

# Eingabe
den = [1, 5, 6, 2]  # p(s) = s^3 + 5 s^2 + 6 s + 2
print("EINGABE den =", den)

# Ausgabe Hurwitz
res_h = hurwitz_kriterium(den)
print("\nAUSGABE Hurwitz:")
print("stabil =", res_h["ergebnis"]["stabil"])
print("Hauptminoren =", res_h["ergebnis"]["hauptminoren"])

# Ausgabe Routh
res_r = routh_kriterium(den)
print("\nAUSGABE Routh:")
print("stabil =", res_r["ergebnis"]["stabil"])
print("Routh-Schema:")
display(res_r["ergebnis"]["schema"])